# Dropout e batch normalization

> **Nota AI Lab — ambiente di lettura.** Il testo seguente è la traduzione del notebook Kaggle originale conservato nel repository. I riferimenti a pulsanti, celle nascoste e invii descrivono l’ambiente Kaggle dell’originale; qui le celle di codice sono visibili. Per eseguire i notebook fuori da Kaggle, segui la guida AI Lab all’ambiente.



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione

Nel mondo del deep learning non esistono soltanto gli strati dense. Ci sono decine di tipi di strati che puoi aggiungere a un modello: prova a sfogliare la [documentazione di Keras](https://www.tensorflow.org/api_docs/python/tf/keras/layers/) per averne un’idea! Alcuni, come gli strati dense, definiscono connessioni tra neuroni; altri eseguono preelaborazioni o trasformazioni di altro tipo.

In questa lezione conosceremo due tipi di strati speciali: non contengono neuroni, ma aggiungono funzionalità che possono essere utili al modello in vari modi. Entrambi sono comuni nelle architetture moderne.

# Dropout

Il primo è lo «strato di dropout», che può aiutare a correggere l’overfitting.

Nella lezione precedente abbiamo spiegato che l’overfitting deriva dall’apprendimento di schemi accidentali nei dati di training. Per riconoscerli, spesso la rete si affida a combinazioni molto specifiche di pesi, una sorta di «complotto» tra pesi. Essendo così specifiche, queste combinazioni tendono a essere fragili: basta rimuovere un elemento e il complotto crolla.

Questa è l’idea del **dropout**. Per spezzare queste combinazioni, a ogni passo di addestramento *disattiviamo* casualmente una frazione delle unità in ingresso a uno strato. Diventa così molto più difficile apprendere gli schemi accidentali dei dati di training. La rete deve invece cercare schemi ampi e generali, le cui configurazioni di pesi tendono a essere più robuste.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/a86utxY.gif" width="600" alt="Animazione di una rete che attraversa diverse configurazioni casuali di dropout.">
<figcaption><center>Qui è stato aggiunto un dropout del 50% tra i due strati nascosti.</center></figcaption>
</figure>

Puoi anche pensare al dropout come alla creazione di una sorta di *ensemble* di reti. Le previsioni non vengono più prodotte da una sola grande rete, ma da un comitato di reti più piccole. I membri del comitato tendono a commettere errori diversi, ma a essere corretti contemporaneamente: il comitato nel suo complesso risulta migliore di ciascun membro. Se conosci le random forest come ensemble di alberi decisionali, l’idea è la stessa.

## Aggiungere il dropout

In Keras, l’argomento `rate` definisce quale percentuale delle unità in ingresso disattivare. Inserisci lo strato `Dropout` subito prima dello strato al quale vuoi applicarlo:

```
keras.Sequential([
    # ...
    layers.Dropout(rate=0.3), # applica il 30% di dropout allo strato successivo
    layers.Dense(16),
    # ...
])
```

# Batch normalization

Il prossimo strato speciale esegue la «normalizzazione per batch», o *batch normalization*, spesso abbreviata in *batchnorm*. Può aiutare a correggere un addestramento lento o instabile.

Con le reti neurali è generalmente una buona idea portare tutti i dati su una scala comune, magari con [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) o [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) di scikit-learn. Il motivo è che SGD sposta i pesi della rete in proporzione all’ampiezza delle attivazioni prodotte dai dati. Feature che tendono a produrre attivazioni di ampiezze molto diverse possono rendere instabile l’addestramento.

Se è utile normalizzare i dati prima di inserirli nella rete, forse normalizzarli anche al suo interno potrebbe essere ancora meglio! Esiste proprio uno strato speciale per farlo: lo **strato di batch normalization**. Esamina ogni batch in arrivo, lo normalizza prima con la sua media e deviazione standard e poi porta i dati su una nuova scala mediante due parametri di riscalamento addestrabili. In effetti, batchnorm esegue una sorta di riscalamento coordinato dei suoi ingressi.

Molto spesso si aggiunge batchnorm per aiutare l’ottimizzazione, anche se a volte può migliorare anche le previsioni. I modelli che la usano tendono a richiedere meno epoche per completare l’addestramento. Inoltre può risolvere vari problemi che fanno «bloccare» l’addestramento. Valuta di aggiungerla ai tuoi modelli, soprattutto se incontri difficoltà durante il training.

## Aggiungere la batch normalization

Sembra possibile usare la batch normalization quasi in qualsiasi punto della rete. Puoi inserirla dopo uno strato:

```
layers.Dense(16, activation='relu'),
layers.BatchNormalization(),
```

oppure tra uno strato e la sua funzione di attivazione:

```
layers.Dense(16),
layers.BatchNormalization(),
layers.Activation('relu'),
```

Se la aggiungi come primo strato della rete, può agire come una sorta di preelaborazione adattiva, sostituendo qualcosa come `StandardScaler` di scikit-learn.

# Esempio: usare dropout e batch normalization

Continuiamo a sviluppare il modello *Red Wine*. Aumenteremo ancora la capacità, aggiungendo però dropout per controllare l’overfitting e batch normalization per accelerare l’ottimizzazione. Questa volta ometteremo anche la standardizzazione dei dati, per mostrare come la batch normalization possa stabilizzare l’addestramento.


In [ ]:

# Configura i grafici.
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
# Imposta i valori predefiniti di Matplotlib.
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)


import pandas as pd
red_wine = pd.read_csv('../input/dl-course-data/red-wine.csv')

# Suddividi i dati in training e validazione.
df_train = red_wine.sample(frac=0.7, random_state=0)
df_valid = red_wine.drop(df_train.index)

# Separa feature e target.
X_train = df_train.drop('quality', axis=1)
X_valid = df_valid.drop('quality', axis=1)
y_train = df_train['quality']
y_valid = df_valid['quality']

Quando aggiungi il dropout, potrebbe essere necessario aumentare il numero di unità negli strati `Dense`.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(1024, activation='relu', input_shape=[11]),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(1),
])

Questa volta non cambia nulla nel modo in cui impostiamo l’addestramento.


In [ ]:
model.compile(
    optimizer='adam',
    loss='mae',
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=256,
    epochs=100,
    verbose=0,
)


# Mostra le curve di apprendimento.
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot();

In genere otterrai prestazioni migliori standardizzando i dati prima di usarli per il training. Essere riusciti a usare comunque i dati grezzi mostra però quanto possa essere efficace la batch normalization sui dataset più difficili.

# Ora tocca a te

Prosegui e [**migliora le previsioni**](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex5.html) sul dataset *Spotify* con il dropout; scopri anche come la batch normalization può aiutare con i dataset difficili.
